# TensorFlow: Generating saliency map for classification predictions

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from common import download_image

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

import tensorflow as tf
import tensorflow_hub as hub
import tf_keras as keras

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

In [ ]:
# The size of images
IMAGE_SIZE = (300, 300)
# The shape of images
IMAGE_SHAPE = IMAGE_SIZE + (3,)
# The number of the classes
N_CLASSES = 1001

## Prepare Model

* Use pre-trained model from TensorFlow Hub
* Specify 'softmax' activation

In [ ]:
URL = 'https://tfhub.dev/google/tf2-preview/inception_v3/classification/4'

model = keras.Sequential([
    hub.KerasLayer(URL, output_shape=[N_CLASSES]),
    keras.layers.Activation('softmax')
])

In [ ]:
# Build model specifying input shape
model.build([None, IMAGE_SHAPE[0], IMAGE_SHAPE[1], IMAGE_SHAPE[2]])

## Download Images

In [ ]:
ROOT_DIR = 'data/saliency_map'

os.makedirs(ROOT_DIR, exist_ok=True)

In [ ]:
# Download sample image to generate predicitons
file01 = download_image(
    url='https://cdn.pixabay.com/photo/2018/02/27/14/11/the-pacific-ocean-3185553_960_720.jpg',
    out=os.path.join(ROOT_DIR, 'image01.jpg'))

## Generate Map

In [ ]:
def get_salience(path, model, class_index, n_classes):
    """
    Generates salience map for given image using trained model
    :param path: the path to the image
    :param model: the trained model
    :param class_index: the index of the class
    :param n_classes: the total amount of classes in the dataset
    :return: the input image and saliency map
    """
    # Preprocess image
    image = cv2.imread(path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, IMAGE_SIZE) / 255.0
    expanded_image = np.expand_dims(image, axis=0)

    # Define the expected output array by one-hot encoding the given class_index
    expected_output = tf.one_hot([class_index] * expanded_image.shape[0], n_classes)

    # 1) Pass an image through trained CNN to get a prediction
    # 2) Calculate the gradient w.r.t. input pixels
    # 3) Evaluate gradients to highlight pixels where the gradient magnitude is higher
    #    (the more important pixels are the higher gradient )
    with tf.GradientTape() as tape:
      inputs = tf.cast(expanded_image, tf.float32)
      tape.watch(inputs)
      predictions = model(inputs)
      loss = tf.keras.losses.categorical_crossentropy(expected_output, predictions)
    gradients = tape.gradient(loss, inputs)

    grayscale_tensor = tf.reduce_sum(tf.abs(gradients), axis=-1)

    # Normalize the pixel values to in the range [0, 255]
    # Formula: 255 * (x - min) / (max - min)
    # Cast the tensor as a tf.uint8
    map = tf.cast(
        255
        * (grayscale_tensor - tf.reduce_min(grayscale_tensor))
        / (tf.reduce_max(grayscale_tensor) - tf.reduce_min(grayscale_tensor)),
        tf.uint8,
    )

    # Remove dimensions with size 1
    map = tf.squeeze(map)

    return image, map

In [ ]:
# Specify the class index corresponding to sample image
CLASS_INDEX = 251

image, salience_map = get_salience(file01, model, CLASS_INDEX, N_CLASSES)

In [ ]:
# Apply salience map on top image
gradient_color = cv2.applyColorMap(salience_map.numpy(), cv2.COLORMAP_HOT)
gradient_color = gradient_color / 255.0
super_imposed = cv2.addWeighted(image, 0.5, gradient_color, 0.5, 0.0)

In [ ]:
# Plot three images
# 1 - origin image
# 2 - salience map
# 3 - image with salience map
plt.figure(figsize=(16, 16))
plt.subplot(2, 2, 1)
plt.axis('off')
plt.imshow(image)
plt.subplot(2, 2, 2)
plt.axis('off')
plt.imshow(salience_map)
plt.subplot(2, 2, (3, 4))
plt.axis('off')
_ = plt.imshow(super_imposed)